In [5]:
import pandas as pd
import polars as pl
from matplotlib import pyplot as plt
import plotnine as pn

In [6]:
import sys
sys.path.append("../")   # path to folder containing the python file

from utils.load_gtf_cgc_dresden import *
from ProteinExpression.load_pr_data import *
from utils.util_functions import *

In [7]:
mapping_table = pd.read_csv("./random_mapping.csv")

sa = pd.read_csv("/omics/odcf/analysis/hipo/hipo_021/outlier_analysis/sample_data/master_drop_sample_annotation_sizeFactorFiltered_0.1.tsv", sep="\t")


In [8]:
spliceosome_genes = pd.read_csv("https://www.genenames.org/cgi-bin/genegroup/download?id=1518&type=branch", sep = "\t")


In [9]:
needed_cols = ["#CHROM", "POS", "REF", "ALT", "germline_Consequence", "germline_IMPACT",  "SpliceAI_pred_DS_AL", "SpliceAI_pred_DS_AG", 
    "SpliceAI_pred_DS_DG", "SpliceAI_pred_DS_DL", "germline_max_spliceai_score", "PID",  "SYMBOL", "Gene"]


dresden_germline_splice = (pl.scan_parquet("/omics/odcf/analysis/hipo/hipo_021/outlier_analysis/vcf/dresden_annotations/germline/germline_damaging_aggregated_all.parquet")
                .filter(pl.col("Gene").is_in(spliceosome_genes["Ensembl gene ID"]))
                .select(needed_cols) 
                 
                .collect(engine="streaming")        
                  ).to_pandas()


dresden_germline_splice = dresden_germline_splice[(dresden_germline_splice["germline_Consequence"].isin(["splice_donor_variant", "splice_acceptor_variant"])) | (dresden_germline_splice["germline_max_spliceai_score"] >= 0.2)]



needed_cols = ["#CHROM", "POS", "REF", "ALT", "somatic_Consequence", "somatic_IMPACT",  "SpliceAI_pred_DS_AL", "SpliceAI_pred_DS_AG", 
    "SpliceAI_pred_DS_DG", "SpliceAI_pred_DS_DL", "somatic_max_spliceai_score", "PID",  "SYMBOL", "Gene"]


dresden_somatic_splice = (pl.scan_parquet("/omics/odcf/analysis/hipo/hipo_021/outlier_analysis/vcf/dresden_annotations/somatic/somatic_damaging_aggregated_all.parquet")
                .filter(pl.col("Gene").is_in(spliceosome_genes["Ensembl gene ID"]))
                .select(needed_cols) 
                 
                .collect(engine="streaming")        
                  ).to_pandas()


dresden_somatic_splice = dresden_somatic_splice[(dresden_somatic_splice["somatic_Consequence"].isin(["splice_donor_variant", "splice_acceptor_variant"])) | (dresden_germline_splice["germline_max_spliceai_score"] >= 0.2)]





/tmp/ipykernel_811562/661464685.py:29: UserWarning: Boolean Series key will be reindexed to match DataFrame index.


In [24]:
dresden_somatic_splice["somatic_IMPACT"].value_counts()

somatic_IMPACT
MODERATE    361
HIGH        119
MODIFIER     41
LOW          22
Name: count, dtype: int64

153
136


In [21]:
merged["IMPACT"].value_counts()

IMPACT
na      1335
HIGH     153
Name: count, dtype: int64

,#CHROM,POS,REF,ALT,somatic_Consequence,somatic_IMPACT,SpliceAI_pred_DS_AL_x,SpliceAI_pred_DS_AG_x,SpliceAI_pred_DS_DG_x,SpliceAI_pred_DS_DL_x,...,Gene,germline_Consequence,germline_IMPACT,SpliceAI_pred_DS_AL_y,SpliceAI_pred_DS_AG_y,SpliceAI_pred_DS_DG_y,SpliceAI_pred_DS_DL_y,germline_max_spliceai_score,SYMBOL_y,IMPACT
0,2,96966941,G,C,NaN,NaN,NaN,NaN,NaN,NaN,...,ENSG00000144028,intron_variant,MODIFIER,0.00,0.00,0.24,0.00,0.24,SNRNP200,na
1,1,25555416,C,A,NaN,NaN,NaN,NaN,NaN,NaN,...,ENSG00000117614,intron_variant,MODIFIER,0.00,0.42,0.00,0.00,0.42,SYF2,na
3,2,96949694,T,C,NaN,NaN,NaN,NaN,NaN,NaN,...,ENSG00000144028,missense_variant,MODERATE,0.00,0.31,0.00,0.00,0.31,SNRNP200,na
4,20,62661143,A,G,NaN,NaN,NaN,NaN,NaN,NaN,...,ENSG00000101161,intron_variant,MODIFIER,0.00,0.00,0.00,0.20,0.20,PRPF6,na
5,10,95451765,C,A,splice_donor_variant,HIGH,0.0,0.0,0.0,0.74,...,ENSG00000148690,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,HIGH
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1483,19,49607639,G,A,NaN,NaN,NaN,NaN,NaN,NaN,...,ENSG00000104852,intron_variant,MODIFIER,0.00,0.00,0.23,0.01,0.23,SNRNP70,na
1484,11,65731531,T,G,NaN,NaN,NaN,NaN,NaN,NaN,...,ENSG00000175467,splice_polypyrimidine_tract_variant&intron_var...,LOW,0.37,0.00,0.00,0.00,0.37,SART1,na
1485,14,73568687,G,A,NaN,NaN,NaN,NaN,NaN,NaN,...,ENSG00000119707,intron_variant,MODIFIER,0.37,0.01,0.00,0.00,0.37,RBM25,na
1486,6,34725704,C,T,NaN,NaN,NaN,NaN,NaN,NaN,...,ENSG00000124562,synonymous_variant,LOW,0.29,0.00,0.00,0.23,0.29,SNRPC,na


In [64]:
dresden_somatic_splice = dresden_somatic_splice.merge(mapping_table, left_on="PID", right_on="nct_id").drop(columns=["PID", "nct_id",])

somatic_counts = dresden_somatic_splice.groupby(['Gene', 'random_id']).size().reset_index(name='n_somatic_splicing_variant')
somatic_counts

,Gene,random_id,n_somatic_splicing_variant
0,ENSG00000003756,BRI3662,1
1,ENSG00000003756,EAA9516,1
2,ENSG00000003756,HIY6488,1
3,ENSG00000003756,HJW6671,2
4,ENSG00000003756,PYZ9132,1
...,...,...,...
502,ENSG00000240682,LNG5339,1
503,ENSG00000240682,VAP5984,1
504,ENSG00000257103,ABV3837,1
505,ENSG00000257103,DIB5249,1


In [10]:
dresden_germline_splice = dresden_germline_splice.merge(mapping_table, left_on="PID", right_on="nct_id").drop(columns=["PID", "nct_id"])

germline_counts = dresden_germline_splice.groupby(['Gene', 'random_id']).size().reset_index(name='n_germline_splicing_variant')
germline_counts

,Gene,random_id,n_germline_splicing_variant
0,ENSG00000003756,AHY8852,1
1,ENSG00000003756,CFD2833,1
2,ENSG00000003756,CXI8998,1
3,ENSG00000003756,DCU0059,1
4,ENSG00000003756,FAV0278,1
...,...,...,...
863,ENSG00000257103,BNP9147,1
864,ENSG00000257103,BNR4176,2
865,ENSG00000257103,CJS5224,1
866,ENSG00000257103,NTF5010,1


,Gene,random_id,n_germline_splicing_variant,n_somatic_splicing_variant
0,ENSG00000003756,AHY8852,1.0,0.0
1,ENSG00000003756,BRI3662,0.0,1.0
2,ENSG00000003756,CFD2833,1.0,0.0
3,ENSG00000003756,CXI8998,1.0,0.0
4,ENSG00000003756,DCU0059,1.0,0.0
...,...,...,...,...
1370,ENSG00000257103,CJS5224,1.0,0.0
1371,ENSG00000257103,DIB5249,0.0,1.0
1372,ENSG00000257103,EIZ6791,0.0,1.0
1373,ENSG00000257103,NTF5010,1.0,0.0


In [2]:
sp_vars = pd.read_parquet("/omics/odcf/analysis/hipo/hipo_021/outlier_analysis/vcf/dresden_annotations/splicosome_variants.parquet")

In [4]:
sp_vars.to_csv("/omics/odcf/analysis/hipo/hipo_021/outlier_analysis/vcf/dresden_annotations/splicosome_variants.csv", index=None)

In [ ]:
sp_vars